<a href="https://colab.research.google.com/github/Suleiman505/Suleiman505/blob/main/summarize_dialogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U datasets==4.8.5 transformers==5.8.0

## ***Installing And Importing Libraries Required***

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [ ]:

dataset=load_dataset("knkarthick/dialogsum")

### **Loading and exploring sample Dialogue Dataset **

In [ ]:
example_indices = [40, 200]

dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

In [ ]:
model_name="google/flan-t5-base"
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

## Loading Model and Tokenizer

In [ ]:
tokeniser=AutoTokenizer.from_pretrained(model_name,use_fast=True)

### Tokenization Demonstration

In [ ]:
Sentence= "Nice to meet you"
sentence_encode=tokeniser(Sentence, return_tensors='pt')
sentence_decode=tokeniser.decode(sentence_encode["input_ids"][0],skip_special_tokens=True)
print(sentence_encode["input_ids"][0])
print("\nDecoded Sentence:")
print(sentence_decode)

## Baseline Summarization Without Prompt Engineering

In [ ]:
for i, index in enumerate(example_indices):
  dialogue=dataset['test'][index]['dialogue']
  summary=dataset['test'][index]['summary']
  inputs=tokeniser(dialogue, return_tensors='pt')
  output=tokeniser.decode(model.generate(inputs["input_ids"], max_new_tokens=50) [0],skip_special_tokens=True)
  print(dash_line)
  print('Example ', i + 1)
  print(dash_line)
  print(f'INPUT PROMPT:\n{dialogue}')
  print(dash_line)
  print(f'BASELINE HUMAN SUMMARY:\n{summary}')
  print(dash_line)
  print(f"Model Generation without prompt Engineering:\n{output}\n")






## Zero-Shot( Prompt Engineering)

In [ ]:
for i, index in enumerate(example_indices):
  dialogue=dataset['test'][index]['dialogue']
  summary=dataset['test'][index]['summary']
  Prompts= f'''
  Summarize the following Sentence
  {dialogue}
  Summary:
    '''
  inputs=tokeniser(Prompts, return_tensors='pt')
  output=tokeniser.decode(model.generate(inputs["input_ids"], max_new_tokens=50) [0],skip_special_tokens=True)
  print(dash_line)
  print('Example ', i + 1)
  print(dash_line)
  print(f'INPUT PROMPT:\n{Prompts}')
  print(dash_line)
  print(f'BASELINE HUMAN SUMMARY:\n{summary}')
  print(dash_line)
  print(f"Model Generation with Zero-shot prompt Engineering:\n{output}\n")

## One-Shot( Prompt Engineering)



In [ ]:
def make_prompt(example_indices_full, example_index_to_summarize):

    prompt = ''

    for index in example_indices_full:

        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}

"""

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

What was going on?
"""

    return prompt




In [ ]:
example_indices_full = [40]
example_index_to_summarize = 200

one_shot_prompt = make_prompt(
    example_indices_full,
    example_index_to_summarize
)

print(one_shot_prompt)




In [ ]:
summary = dataset['test'][example_index_to_summarize]['summary']

inputs = tokeniser(one_shot_prompt, return_tensors='pt')

output = tokeniser.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

## Few-Shot( Prompt Engineering)

In [ ]:
def make_prompt(example_indices_full, example_index_to_summarize):

    prompt = ""

    for index in example_indices_full:

        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}

"""

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

What was going on?
"""

    return prompt


# Three examples instead of one
example_indices_full = [40, 80, 120]
example_index_to_summarize = 200

few_shot_prompt = make_prompt(
    example_indices_full,
    example_index_to_summarize
)

print(few_shot_prompt)

In [ ]:
summary = dataset['test'][example_index_to_summarize]['summary']

inputs = tokeniser(few_shot_prompt, return_tensors='pt')

output = tokeniser.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')